# 01 · Calibração na L4 e ponte fp32 × AMP

**Sessão P1 · ambiente L4.** Roda uma vez, antes de qualquer rodada longa (~2–3 h de L4). Se a sessão cair, abra de novo e **Executar tudo**: cada parte grava o que já fez em `resultados/calibracao_l4.json`, e o treino da ponte retoma por época.

| Parte | O que mede/decide |
|---|---|
| 1 | contagem de parâmetros das cabeças, avisos da Albumentations |
| 2 | **teste de matar e retomar** (histórico e predições iguais) |
| 3 | teste de plugin (registrar cabeça/config sem mudar o hash do protocolo) |
| 4 | **ponte fp32 × AMP** → `decisao_precisao` (vale para TODAS as rodadas) |
| 5 | vazão, memória e horas por rodada 5-fold de cada modelo |
| 6 | custo de gravar checkpoint no Drive |
| 7 | cabeças W1/W5 (Wav-KAN) e *smoke* da multitarefa, **se os arquivos existirem** |
| 8 | **unidades de computação**: un/h e unidades por rodada → o que cabe no saldo |

**Antes de começar:** anote o saldo de unidades e o consumo/h do painel *Recursos* (canto superior direito do Colab) em `SALDO_INICIAL` e `TAXA_PAINEL`. Ao terminar, preencha `SALDO_FINAL` na parte 8 e rode só essa célula.

**Regra da ponte (pré-registrada).** Primeiro tenta reconstruir a dobra 1 antiga. Se a impressão digital (496/636/459/465/498) conferir, a faixa de acurácia é 79,84 ± 3 pp; se não, usa a dobra 0 nova com a faixa das dobras antigas ± 3 pp (76,84–91,89 %). O AMP é adotado se |Δacc| ≤ 2 pp, |ΔAUC| ≤ 0,015 e as duas acurácias estiverem na faixa. Se só a acurácia falhar, roda uma 2ª execução AMP (semente 43); se ela também falhar, ou se a AUC falhar, tudo roda em fp32. Se o próprio fp32 sair da faixa, o notebook **para** e você avisa o P1.

In [ ]:
# ==== Configuração desta sessão ====
DIR_DRIVE = "/content/drive/MyDrive/Colab Notebooks/tcc-final"  # pasta tcc-final (conta padrão, authuser=0)
RAIZ_TRABALHO = "/content"   # onde o dataset é clonado e o cache é gerado (disco local, temporário)
SHA_PROTOCOLO_ESPERADO = "b7cd9baca53edd96c855cca10914fc258efa8ddfd0937cc0d3e9572de5ef9e13"
ABA = "A"                  # dono das travas desta aba (numa 2ª aba simultânea use "B")
PERMITIR_RESERVA = False   # a calibração tem de ser na L4
SALDO_INICIAL = None       # unidades de computação no painel "Recursos" ao começar (ex.: 98.5)
TAXA_PAINEL = None         # "consumo por hora" mostrado no painel (ex.: 4.82)

In [ ]:
# Dependências fixadas (rode ANTES de importar qualquer coisa da albumentations)
!pip install -q "albumentations==2.0.8" "git+https://github.com/Blealtan/efficient-kan@7b6ce1c87f18c8bc90c208f6b494042344216b11"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import sys, os, json, time
from pathlib import Path
import numpy as np, pandas as pd, torch
sys.path.insert(0, DIR_DRIVE)
import protocolo_comum as pc

pc.checar_sentinela_drive(DIR_DRIVE)   # PARA aqui se o Colab estiver logado em outra conta
assert pc.sha256_protocolo() == SHA_PROTOCOLO_ESPERADO, (
    "O protocolo_comum.py do Drive não é o publicado. Não rode nada; avise o P1.")
print("Dependências:", pc.checar_dependencias())
print("Protocolo", pc.PROTOCOLO_VERSAO, "| SHA-256", pc.sha256_protocolo()[:16], "…")
GPU = pc.exigir_gpu(permitir_reserva=PERMITIR_RESERVA)   # rodadas comparáveis só na L4
pc.definir_id_sessao(ABA)
print("GPU:", GPU["nome"], "| comparável:", GPU["comparavel"], "| dono das travas:", pc.ID_SESSAO)

In [ ]:
# Clona o dataset e o leguwoi nos commits fixados e gera o cache 224×224 só nesta sessão (~3–5 min)
sess = pc.preparar_sessao(
    RAIZ_TRABALHO,
    caminho_dobras=f"{DIR_DRIVE}/dados/{pc.ARQUIVO_DOBRAS}",
    caminho_meta=f"{DIR_DRIVE}/dados/{pc.ARQUIVO_META_DOBRAS}",
)
R, C = f"{DIR_DRIVE}/resultados", f"{DIR_DRIVE}/checkpoints"
RT, CT = f"{DIR_DRIVE}/resultados_teste", f"{DIR_DRIVE}/checkpoints_teste"
print(json.dumps(sess.meta, indent=1))
if sess.meta.get("cache_confere") is False:
    print("⚠️ O cache diverge do registrado no 00 (versão do OpenCV?). Divergência:", sess.meta.get("cache_divergencia"))

In [ ]:
ARQ_CAL = Path(R) / "calibracao_l4.json"
Path(R).mkdir(parents=True, exist_ok=True)
cal = json.loads(ARQ_CAL.read_text()) if ARQ_CAL.exists() else {}
def salvar_cal():
    pc.gravar_json_atomico(cal, ARQ_CAL)
cal.setdefault("inicio", {"quando": pc.agora_iso(), "t": time.time(), "saldo": SALDO_INICIAL, "taxa_painel": TAXA_PAINEL})
cal.setdefault("sessoes", []).append({"quando": pc.agora_iso(), "gpu": GPU, "id": pc.ID_SESSAO})
cal["protocolo_sha256"] = pc.sha256_protocolo()
cal["ambiente"] = pc.info_ambiente()
cal["dados"] = sess.meta
salvar_cal()

## 1. Parâmetros e avisos

In [ ]:
cont = {k: sum(p.numel() for p in pc._CABECAS[c](2048, 2).parameters())
        for k, c in [("a", "linear"), ("b", "mlp128"), ("c", "mlp_iso"), ("d", "kan"), ("e", "kan_pequena")]}
print(cont)
assert cont == {"a": 4098, "b": 262530, "c": 3935871, "d": 3936000, "e": 246000}
avisos = pc.checar_avisos_transforms(sess.cache[:5])
assert not avisos, avisos
cal["parametros_cabecas"] = cont
salvar_cal()

In [ ]:
m = pc.ModeloProtocolo(pc.obter_config("resnet152__kan"))
par = pc.contar_parametros(m)
print(par, m.info_backbone)
assert par["backbone"] == 58_143_808
cal["parametros_resnet152_kan"] = par
salvar_cal()
del m

## 2. Teste de matar e retomar

In [ ]:
import shutil
if not cal.get("retomada", {}).get("ok"):
    torch.backends.cudnn.deterministic = True
    for e in ("t01_ref", "t01_interrompido"):
        shutil.rmtree(f"{RT}/{e}", ignore_errors=True); shutil.rmtree(f"{CT}/{e}", ignore_errors=True)
    kw = dict(precisao="amp", limite_epocas=(1, 2), limite_imagens=128, espera_trava_s=0)
    u_ref = {"experimento": "t01_ref", "config": "resnet152__kan", "semente": 42, "dobra": 0}
    u_int = {**u_ref, "experimento": "t01_interrompido"}
    pc.executar_unidade(sess, u_ref, RT, CT, **kw)
    for parar in (1, 2):   # interrompe no fim da Fase 1 e no meio da Fase 2
        try:
            pc.executar_unidade(sess, u_int, RT, CT, parar_apos_epoca=parar, **kw)
        except pc.InterrupcaoSimulada as e:
            print("interrupção simulada:", e)
    pc.executar_unidade(sess, u_int, RT, CT, **kw)
    base = "resnet152__kan/s42/dobra0"
    h1, h2 = (pd.read_csv(f"{RT}/{e}/{base}/historico.csv") for e in ("t01_ref", "t01_interrompido"))
    p1, p2 = (pd.read_csv(f"{RT}/{e}/{base}/pred_teste.csv") for e in ("t01_ref", "t01_interrompido"))
    dif_perda = float(np.abs(h1["perda"] - h2["perda"]).max())
    dif_prob = float(np.abs(p1["p_es_1"] - p2["p_es_1"]).max())
    cal["retomada"] = {"max_dif_perda": dif_perda, "max_dif_prob": dif_prob, "ok": dif_perda < 1e-3 and dif_prob < 1e-3}
    torch.backends.cudnn.deterministic = False
    salvar_cal()
print(cal["retomada"])
assert cal["retomada"]["ok"], "A retomada não reproduziu o treino — avise o P1"

## 3. Teste de plugin (registra cabeça e config sem mudar o hash do protocolo)

In [ ]:
plug = Path(RAIZ_TRABALHO) / "plugin_exemplo_01.py"
plug.write_text(
    "import torch.nn as nn\n"
    "import protocolo_comum as pc\n"
    "pc.registrar_cabeca('exemplo_01', lambda i, n, **k: nn.Sequential(nn.Linear(i, 16), nn.GELU(), nn.Linear(16, n)), sobrescrever=True)\n"
    "pc.registrar_config('resnet152__exemplo_01', {'backbone': 'resnet152', 'cabecas': {'es': 'exemplo_01'}}, sobrescrever=True)\n"
)
sha_antes = pc.sha256_protocolo()
pc.carregar_plugin(plug)
u = {"experimento": "t01_plugin", "config": "resnet152__exemplo_01", "semente": 42, "dobra": 0}
r = pc.executar_unidade(sess, u, RT, CT, precisao="amp", limite_epocas=(1, 1), limite_imagens=64, espera_trava_s=0) \
    or pc.ler_metricas_unidade(RT, u)
cal["plugin"] = {"ok": pc.sha256_protocolo() == sha_antes and "es" in r["metricas"],
                 "plugins_registrados": r["plugins"]}
salvar_cal()
print(cal["plugin"])
assert cal["plugin"]["ok"]

## 4. Ponte fp32 × AMP (decide a precisão de todas as rodadas)

In [ ]:
# A escolha (dobra antiga ou nova) é feita UMA vez e fica gravada; sessões seguintes a reaproveitam.
arq_antigo = Path(DIR_DRIVE) / "dados" / "ponte_dobras_antigas.csv"   # só índices de dobra, nenhuma imagem
if cal.get("ponte_origem") == "antiga" or (cal.get("ponte_origem") is None and arq_antigo.exists()):
    antigo = pd.read_csv(arq_antigo)
    dig = pc.conferir_impressao_digital(antigo)
    cal["ponte_origem"] = "antiga"
elif cal.get("ponte_origem") == "nova":
    dig = cal.get("ponte_impressao_digital")
else:
    antigo = pc.dobras_antigas(sess.raiz_dataset)
    dig = pc.conferir_impressao_digital(antigo)
    if dig["confere"]:
        antigo.to_csv(arq_antigo, index=False)
    cal["ponte_origem"] = "antiga" if dig["confere"] else "nova"
cal["ponte_impressao_digital"] = dig
salvar_cal()
print(dig)
if cal["ponte_origem"] == "antiga":
    sess_ponte = pc.Sessao(df=sess.df.merge(antigo[["rel_path", "fold_antiga"]], on="rel_path", how="left", validate="one_to_one"),
                           cache=sess.cache, raiz_dataset=sess.raiz_dataset, meta=sess.meta)
    COL, FAIXA, ORIGEM = "fold_antiga", pc.FAIXA_PONTE_ANTIGA, "dobra 1 antiga (impressão digital confere)"
else:
    sess_ponte, COL, FAIXA, ORIGEM = sess, "fold", pc.FAIXA_PONTE_NOVA, "dobra 0 nova (ordem antiga não reproduzível nesta máquina)"
print("Ponte na", ORIGEM, "| faixa de acurácia", FAIXA)

def rodar_ponte(experimento, semente, precisao):
    u = {"experimento": experimento, "config": "resnet152__kan", "semente": semente, "dobra": 0, "coluna_dobra": COL}
    r = pc.executar_unidade(sess_ponte, u, R, C, precisao=precisao) or pc.ler_metricas_unidade(R, u)
    m = r["metricas"]["es"]
    return {"acc": 100 * m["acc"], "auc": m["auc"], "f1_macro": m["f1_macro"], "treino_min": r["tempos"]["treino_s"] / 60,
            "precisao": r["unidade"]["precisao"], "semente": semente, "n_teste": r["n_teste"]}

res_fp32 = rodar_ponte("ponte_fp32", 42, "fp32")
res_amp = rodar_ponte("ponte_amp", 42, "amp")
dec = pc.avaliar_ponte(res_fp32, res_amp, faixa_acc=FAIXA)
if dec["decisao"] == "rodar_amp2":
    print("Só a acurácia falhou; rodando a 2ª execução AMP (semente 43)…")
    dec = pc.avaliar_ponte(res_fp32, res_amp, rodar_ponte("ponte_amp", 43, "amp"), faixa_acc=FAIXA)
dec.update(origem=ORIGEM, impressao_digital=dig)
cal["ponte"] = dec
cal["decisao_precisao"] = {"amp": "amp", "fp32": "fp32"}.get(dec["decisao"])
salvar_cal()
print(json.dumps({k: dec[k] for k in ("decisao", "motivo", "origem", "fp32", "amp", "amp2")}, indent=1, default=str))
if dec["decisao"] == "parar":
    raise RuntimeError("PONTE: o próprio fp32 saiu da faixa — o problema não é numérico. Pare e avise o P1.")
print("\n➡️ Precisão de TODAS as rodadas:", cal["decisao_precisao"])

## 5. Vazão, memória e horas por rodada 5-fold

In [ ]:
PREC = cal["decisao_precisao"]
modelos = [pc.CONFIGS_EXP1[k] for k in "dcbae"] + [pc.CONFIGS_EXP2[b] for b in pc.BASELINES]
cal.setdefault("vazao", {})
for nome in modelos + ["resnet152__kan@fp32"]:
    if nome in cal["vazao"]:
        continue
    cfg_nome, prec = (nome.split("@")[0], "fp32") if "@" in nome else (nome, PREC)
    v = pc.medir_vazao(sess, cfg_nome, precisao=prec)
    v["estimativa"] = pc.estimar_rodada(v)
    cal["vazao"][nome] = v
    salvar_cal()
for nome, v in cal["vazao"].items():
    print(f"{nome:30s} F1 {v['fase1']['img_s']:7.1f} img/s | F2 {v['fase2']['img_s']:6.1f} img/s | "
          f"mem F2 {v['fase2'].get('mem_pico_gb', float('nan')):5.1f} GB | {v['estimativa']['horas_rodada_5fold']:.2f} h/rodada")

## 6. Custo de gravar checkpoint no Drive

In [ ]:
cal.setdefault("gravacao", {})
for nome in ("resnet152__kan", "vgg16__linear"):
    if nome in cal["gravacao"]:
        continue
    m = pc.ModeloProtocolo(pc.obter_config(nome))
    cal["gravacao"][nome] = pc.medir_gravacao(m, f"{DIR_DRIVE}/checkpoints_teste")
    del m
    salvar_cal()
print(cal["gravacao"])
g = cal["gravacao"]["resnet152__kan"]
seg_ckpt = g["s_destino"] + g["s_local"]
s_epoca_f2 = cal["vazao"][pc.CONFIGS_EXP1["d"]]["estimativa"]["s_epoca_f2"]
cal["recomendacao_ckpt_intervalo_min"] = 0.0 if seg_ckpt <= 0.1 * s_epoca_f2 else 5.0
salvar_cal()
print(f"checkpoint {seg_ckpt:.1f}s vs época F2 {s_epoca_f2:.1f}s -> CKPT_INTERVALO_MIN recomendado =",
      cal["recomendacao_ckpt_intervalo_min"])

## 7. Cabeças W1/W5 (Wav-KAN) e *smoke* da multitarefa — só se os arquivos já existirem

In [ ]:
NATIVAS = {"linear", "mlp128", "mlp_iso", "kan", "kan_pequena", "conv1x1_squeezenet", "exemplo_01"}
p_wav = Path(DIR_DRIVE) / "cabecas_wavkan.py"
if p_wav.exists():
    pc.carregar_plugin(p_wav)
    externas = sorted(set(pc.cabecas_registradas()) - NATIVAS)
    alvo = [c for c in externas if c.upper() in ("W1", "W5")] or externas
    print("Cabeças do plugin:", externas, "| medindo:", alvo)
    cal.setdefault("wavkan", {})
    for cab in alvo:
        nome_cfg = f"resnet152__{cab}__calib01"
        if nome_cfg not in pc.configs_registradas():
            pc.registrar_config(nome_cfg, {"backbone": "resnet152", "cabecas": {"es": cab}})
        v = pc.medir_vazao(sess, nome_cfg, precisao=cal["decisao_precisao"])
        v["estimativa"] = pc.estimar_rodada(v)
        v["plugin_sha256"] = pc.PLUGINS_CARREGADOS[p_wav.stem]["sha256"]
        cal["wavkan"][cab] = v
        salvar_cal()
        print(f"{cab}: params cabeça {v['parametros']['cabecas_total']:,} | mem F2 {v['fase2'].get('mem_pico_gb')} GB | "
              f"{v['estimativa']['horas_rodada_5fold']:.2f} h/rodada")
else:
    print("cabecas_wavkan.py ainda não está no Drive — pulei (rode esta célula de novo quando existir).")

In [ ]:
p_mt = Path(DIR_DRIVE) / "multitarefa.py"
if p_mt.exists():
    pc.carregar_plugin(p_mt)
    cfgs_mt = [c for c in pc.configs_registradas() if set(pc.obter_config(c)["cabecas"]) == {"es", "dist"}]
    print("Configs multitarefa registradas:", cfgs_mt)
    if cfgs_mt:
        u = {"experimento": "t01_multitarefa", "config": cfgs_mt[0], "semente": 42, "dobra": 0}
        r = pc.executar_unidade(sess, u, RT, CT, precisao=cal["decisao_precisao"], limite_epocas=(1, 1),
                                limite_imagens=128, espera_trava_s=0) or pc.ler_metricas_unidade(RT, u)
        h = pd.read_csv(Path(RT) / "t01_multitarefa" / cfgs_mt[0] / "s42" / "dobra0" / "historico.csv")
        v = pc.medir_vazao(sess, cfgs_mt[0], precisao=cal["decisao_precisao"])
        cal["multitarefa_smoke"] = {
            "config": cfgs_mt[0],
            "ok": set(r["metricas"]) == {"es", "dist"} and bool(np.isfinite(h["perda"]).all()),
            "auc": {t: m["auc"] for t, m in r["metricas"].items()},
            "mem_pico_gb_f2": v["fase2"].get("mem_pico_gb"),
            "horas_rodada_5fold": pc.estimar_rodada(v)["horas_rodada_5fold"],
            "plugin_sha256": pc.PLUGINS_CARREGADOS[p_mt.stem]["sha256"],
        }
        salvar_cal()
        print(cal["multitarefa_smoke"])
else:
    print("multitarefa.py ainda não está no Drive — pulei (rode esta célula de novo quando existir).")

## 8. Unidades de computação: o que cabe no saldo

Preencha `SALDO_FINAL` com o saldo do painel *Recursos* **agora** e rode a célula. Ela calcula as unidades por hora reais (Δsaldo/Δtempo). Se você não anotou os saldos, ela usa `TAXA_PAINEL`.

In [ ]:
SALDO_FINAL = None   # saldo do painel "Recursos" ao terminar (ex.: 88.1)

ini = cal["inicio"]
horas = (time.time() - ini["t"]) / 3600
if SALDO_FINAL is not None and ini.get("saldo") is not None and horas > 0.2:
    un_h = (ini["saldo"] - SALDO_FINAL) / horas
    fonte = f"Δsaldo/Δtempo ({ini['saldo']} → {SALDO_FINAL} em {horas:.2f} h)"
else:
    un_h = ini.get("taxa_painel") or TAXA_PAINEL
    fonte = "valor do painel"
cal["unidades_por_hora"] = {"valor": un_h, "fonte": fonte, "taxa_painel": ini.get("taxa_painel") or TAXA_PAINEL}
fator = 1.10   # +10 % para setup da sessão, travas e gravação de checkpoint
h = {n: v["estimativa"]["horas_rodada_5fold"] * fator for n, v in cal["vazao"].items()}
E1, E2 = pc.CONFIGS_EXP1, pc.CONFIGS_EXP2
etapas = [
    ("cabeças (d,c,b,a) s42", [E1[k] for k in "dcba"]),
    ("baselines baratos/médios + Swin-B", [E2[b] for b in pc.BASELINES[:8]]),
    ("sementes 43/44 de (b,c,d)", [E1[k] for k in "dcb"] * 2),
    ("(a) sementes 43/44", [E1["a"]] * 2),
    ("pesados (EffNetV2-L, VGG16, RegNetY-16GF)", [E2[b] for b in pc.BASELINES[8:]]),
    ("cabeça (e) opcional", [E1["e"]]),
]
saldo = SALDO_FINAL if SALDO_FINAL is not None else None
linhas, acum = [], 0.0
for nome, lista in etapas:
    hh = sum(h[x] for x in lista)
    acum += hh
    un = hh * un_h if un_h else None
    linhas.append({"etapa": nome, "horas_L4": round(hh, 2), "unidades": round(un, 1) if un else None,
                   "horas_acum": round(acum, 2), "unidades_acum": round(acum * un_h, 1) if un_h else None,
                   "cabe_no_saldo": (acum * un_h <= saldo) if (un_h and saldo is not None) else None})
tab = pd.DataFrame(linhas)
cal["orcamento"] = {"fator_overhead": fator, "etapas": linhas, "saldo_referencia": saldo}
salvar_cal()
print("unidades/hora:", un_h, "|", fonte)
display(tab)
print("\n⚠️ Desconecte o ambiente agora (Ambiente de execução → Desconectar e excluir) para não consumir unidades parado.")